# Classify all beach cells with best ConvNeXt Tiny

This notebook reads all complete cell matrices from `/Volumes/BACKUPS/Piotr_beach/beach_all`, loads the best saved ConvNeXt Tiny model, and writes per-date classification CSV files to `Results/classifications/all_cels`.

It does not copy any input `.txt` files from the external drive.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Source.inference import (
    CellInferenceDataset,
    discover_complete_cells_from_base_suffix,
    discover_date_directories,
    load_convnext_for_inference,
    load_inference_metadata,
    predict_cells,
)
from Source.training import get_device

/Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SOURCE_ROOT = Path("/Volumes/BACKUPS/Piotr_beach/beach_all")
MODEL_RESULTS_DIR = PROJECT_ROOT / "Results" / "convnext_tiny"
OUTPUT_ROOT = PROJECT_ROOT / "Results" / "classifications" / "all_cels"

CHECKPOINT_PATH = MODEL_RESULTS_DIR / "convnext_tiny_best.pt"
INFERENCE_METADATA_PATH = MODEL_RESULTS_DIR / "inference_metadata.json"

BATCH_SIZE = 128
NUM_WORKERS = 2
BASE_SUFFIX = "r"

print(f"Project root:       {PROJECT_ROOT}")
print(f"Source root:        {SOURCE_ROOT}")
print(f"Model checkpoint:   {CHECKPOINT_PATH}")
print(f"Inference metadata: {INFERENCE_METADATA_PATH}")
print(f"Output root:        {OUTPUT_ROOT}")

Project root:       /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software
Source root:        /Volumes/BACKUPS/Piotr_beach/beach_all
Model checkpoint:   /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/Results/convnext_tiny/convnext_tiny_best.pt
Inference metadata: /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/Results/convnext_tiny/inference_metadata.json
Output root:        /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/Results/classifications/all_cels


In [3]:
if not SOURCE_ROOT.exists():
    raise FileNotFoundError(f"External source directory not found: {SOURCE_ROOT}")

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"ConvNeXt checkpoint not found: {CHECKPOINT_PATH}")

if not INFERENCE_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Inference metadata not found: {INFERENCE_METADATA_PATH}. "
        "Rerun the ConvNeXt training notebook so it saves inference_metadata.json."
    )

metadata = load_inference_metadata(INFERENCE_METADATA_PATH)
idx_to_class_code = {
    int(index): str(class_code)
    for index, class_code in metadata["idx_to_class_code"].items()
}

print(f"Image size:       {metadata['image_size']}")
print(f"Feature suffixes: {metadata['feature_suffixes']}")
print(f"Class mapping:    {idx_to_class_code}")

Image size:       64
Feature suffixes: ['r', 'g', 'b', 's', 'd']
Class mapping:    {0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9'}


In [4]:
device = get_device()
model = load_convnext_for_inference(CHECKPOINT_PATH, metadata, device=device)
print(f"Device: {device}")

Device: mps


In [5]:
date_dirs = discover_date_directories(SOURCE_ROOT)
print(f"Date folders found: {len(date_dirs)}")
print("This notebook uses a single-pass workflow: each date folder is scanned once using BASE_SUFFIX, then classified immediately.")

Date folders found: 15
This notebook uses a single-pass workflow: each date folder is scanned once using BASE_SUFFIX, then classified immediately.


In [6]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

written_rows = []
for date_dir in tqdm(date_dirs, desc="Classifying date folders"):
    date = date_dir.name
    cell_ids, incomplete_count = discover_complete_cells_from_base_suffix(
        date_dir,
        feature_suffixes=metadata["feature_suffixes"],
        base_suffix=BASE_SUFFIX,
    )
    if not cell_ids:
        written_rows.append(
            {
                "date": date,
                "classified_cells": 0,
                "incomplete_candidates": incomplete_count,
                "output_path": None,
            }
        )
        continue

    dataset = CellInferenceDataset(
        source_root=SOURCE_ROOT,
        date=date,
        cell_ids=cell_ids,
        image_size=metadata["image_size"],
        mean=metadata["channel_mean"],
        std=metadata["channel_std"],
        feature_suffixes=metadata["feature_suffixes"],
    )
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )

    progress_description = f"Classifying {date} ({len(cell_ids):,} cells)"
    predictions = predict_cells(
        model,
        loader,
        idx_to_class_code,
        device,
        progress_description=progress_description,
    )
    predictions.insert(0, "date", date)
    predictions = predictions[["date", "cell_id", "predicted_class"]]

    output_dir = OUTPUT_ROOT / date
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "classification.csv"
    predictions.to_csv(output_path, index=False)

    written_rows.append(
        {
            "date": date,
            "classified_cells": len(predictions),
            "incomplete_candidates": incomplete_count,
            "output_path": output_path,
        }
    )
    print(
        f"{date}: classified {len(predictions):,} cells, "
        f"skipped {incomplete_count:,} incomplete candidates, "
        f"saved {output_path}"
    )

written_summary = pd.DataFrame(written_rows)
written_summary

Classifying date folders:   7%|▋         | 1/15 [06:20<1:28:51, 380.82s/it]

20180722: classified 46,483 cells, skipped 0 incomplete candidates, saved /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/Results/classifications/all_cels/20180722/classification.csv


Classifying date folders:  13%|█▎        | 2/15 [16:36<1:52:23, 518.74s/it]

20180731: classified 54,304 cells, skipped 0 incomplete candidates, saved /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/Results/classifications/all_cels/20180731/classification.csv


Classifying date folders:  20%|██        | 3/15 [29:58<2:09:40, 648.41s/it]

...

Classifying date folders:  93%|█████████▎| 14/15 [2:35:34<10:58, 658.16s/it]

20240623: classified 72,043 cells, skipped 0 incomplete candidates, saved /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/Results/classifications/all_cels/20240623/classification.csv


Classifying date folders: 100%|██████████| 15/15 [2:47:50<00:00, 671.35s/it]

20240806: classified 58,149 cells, skipped 0 incomplete candidates, saved /Users/piotr/Praca/Nauka/publikacje/Beach_classification/Beach_classification_software/Results/classifications/all_cels/20240806/classification.csv


,date,classified_cells,incomplete_candidates,output_path
0,20180722,46483,0,/Users/piotr/Praca/Nauka/publikacje/Beach_clas...
1,20180731,54304,0,/Users/piotr/Praca/Nauka/publikacje/Beach_clas...
2,20180812,67022,0,/Users/piotr/Praca/Nauka/publikacje/Beach_clas...
3,20180826,62815,0,/Users/piotr/Praca/Nauka/publikacje/Beach_clas...
4,20180910,84196,0,/Users/piotr/Praca/Nauka/publikacje/Beach_clas...
5,20180915,52432,0,/Users/piotr/Praca/Nauka/publikacje/Beach_clas...
6,20190822,61357,0,/Users/piotr/Praca/Nauka/publikacje/Beach_clas...
7,20220826,74975,0,/Users/piotr/Praca/Nauka/publikacje/Beach_clas...
8,20220910,50070,0,/Users/piotr/Praca/Nauka/publikacje/Beach_clas...
9,20220924,50765,0,/Users/piotr/Praca/Nauka/publikacje/Beach_clas...
